In [ ]:
!pip install transformers
import pandas as pd

In [ ]:
df = pd.read_csv("train.csv",on_bad_lines='skip', engine='python')
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [ ]:
df.shape

(48645, 8)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48645 entries, 0 to 48644
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             48645 non-null  object
 1   comment_text   48645 non-null  object
 2   toxic          48645 non-null  int64 
 3   severe_toxic   48645 non-null  int64 
 4   obscene        48645 non-null  int64 
 5   threat         48645 non-null  int64 
 6   insult         48645 non-null  int64 
 7   identity_hate  48645 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 3.0+ MB


In [ ]:
df.drop(columns=['id'],axis=0,inplace=True)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:

X =list(df['comment_text'])

In [ ]:
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
y = df[label_cols].values.astype(int)

In [ ]:
y

array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       ...,
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0]])

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)

In [ ]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
train_encodings = tokenizer(list(X_train), truncation=True, padding=True,max_length=128)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True,max_length=128)

In [ ]:
import torch

class ToxicCommentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = ToxicCommentDataset(train_encodings, y_train)
test_dataset = ToxicCommentDataset(test_encodings, y_test)

In [ ]:
from transformers import DistilBertForSequenceClassification


model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=6,
    problem_type="multi_label_classification"
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    sigmoid = 1 / (1 + np.exp(-predictions))
    preds = (sigmoid > 0.5).astype(int)

    f1 = f1_score(labels, preds, average='macro')
    roc_auc = roc_auc_score(labels, sigmoid, average='macro')

    return {
        'f1': f1,
        'roc_auc': roc_auc
    }



In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Roc Auc
1,0.049992,0.044244,0.579963,0.986932


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7299, training_loss=0.04550296860306961, metrics={'train_runtime': 1746.4869, 'train_samples_per_second': 66.847, 'train_steps_per_second': 4.179, 'total_flos': 3866601766311936.0, 'train_loss': 0.04550296860306961, 'epoch': 3.0})

TrainOutput(global_step=7299, training_loss=0.04550296860306961, metrics={'train_runtime': 1746.4869, 'train_samples_per_second': 66.847, 'train_steps_per_second': 4.179, 'total_flos': 3866601766311936.0, 'train_loss': 0.04550296860306961, 'epoch': 3.0})

In [ ]:

model.save_pretrained('./toxic_model')
tokenizer.save_pretrained('./toxic_model')
import shutil

shutil.make_archive('toxic_model', 'zip', './toxic_model')

print("Model successfully saved and zipped!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved and zipped!
